# ⚖️ Regulus — AI Governance Standards Lookup

**Describe an AI issue or system in plain language; Regulus returns the regulatory provisions that apply — across frameworks, each with a source citation, cited cross-framework references, and an optional grounded interpretation.**

Regulus is, in one line, **RAG + a knowledge network**: retrieval over *real* regulatory text, enriched by a *cited* cross-framework graph, optionally interpreted by an LLM. It is built for AI governance and model-risk work, where the question is rarely *"what text is similar?"* but *"**which requirement applies, how does it map across frameworks, and can you show me the source?**"*

It is built on the [Geometric Knowledge Network (GKN)](https://github.com/minw0607/geometric_knowledge_network).

### This notebook, step by step
1. **Setup** — imports.
2. **Configure** — choose the standards, the retriever, and describe the AI system under review.
3. **Launch** — build the system (corpus + knowledge graph + interpreter) and show basic info.
4. **Scenarios** — realistic observations/questions to test.
5. **Retrieve & answer** — applicable provisions, then a grounded, cited interpretation.
6. **Visualize** — the regulatory neighborhood of the issue.

> **Guardrail:** every provision is a real, cited unit of an official framework; cross-framework crosswalks are **curated and cited, never invented**; and the LLM **interprets — it does not author regulation**.

## 1. Setup

In [ ]:
# Auto-reload edited modules; no kernel restart needed after code changes.
try:
    _ip = get_ipython(); _ip.run_line_magic('load_ext', 'autoreload'); _ip.run_line_magic('autoreload', '2')
except Exception:
    pass

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))   # make `regulus` importable

from regulus import demo
demo.ensure_gkn()   # use installed GKN, or the local sibling checkout

## 2. Configure the system

Set your choices below.

- **`STANDARDS`** — which frameworks to load (comment any out).
- **`RETRIEVER`** — `'tfidf'` (no keys) or `'embedding'` (sharper; uses your Azure/OpenAI keys, which Regulus reuses automatically from the GKN `.env`).
- **`TARGET_SYSTEM`** — a description of the AI system under review; the interpreter tailors its answer to it.

*Risk categories* are the seven NIST trustworthiness characteristics (valid & reliable, safe, secure & resilient, accountable & transparent, explainable, privacy-enhanced, fair) — applied automatically; to customize, edit `risk.py`.

In [ ]:
STANDARDS = [
    'eu_ai_act',      # EU AI Act — 113 articles
    'nist_ai_rmf',    # NIST AI RMF 1.0 — 72 subcategories
    'nist_ai_600_1',  # NIST AI 600-1 (GenAI Profile) — 49 action groups
    'oecd_ai',        # OECD AI Principles — 10
    'iso_42001',      # ISO/IEC 42001 — clause structure (reference)
]

RETRIEVER = 'tfidf'   # 'tfidf' (no keys) or 'embedding' (recommended if you have Azure/OpenAI keys)

TARGET_SYSTEM = (
    'A generative-AI customer-service assistant for a retail bank. It answers customer '
    'questions about mortgages, savings, and credit cards by retrieving from internal '
    'product documentation, and is live to the public.'
)

## 3. Launch the system

`demo.launch()` loads the chosen standards into citable provisions, builds the cited knowledge graph, and wires up the interpreter. `info()` reports what's loaded, the retriever, and whether the LLM is configured (it reuses your GKN Azure/OpenAI credentials automatically).

In [ ]:
reg = demo.launch(standards=STANDARDS, retriever=RETRIEVER, target_system=TARGET_SYSTEM)
reg.info()

In [ ]:
# The knowledge network at a glance: frameworks linked by the number of cited crosswalks.
_ = reg.framework_map()   # needs matplotlib

## 4. Test scenarios

Realistic, multi-issue observations. Pick one (or write your own) to run through the system.

In [ ]:
SCENARIOS = {
    'GenAI banking assistant': (
        'We launched a generative-AI assistant on our retail bank website that answers customer '
        'questions about mortgages, savings, and credit cards using retrieval over internal product '
        'documentation. In testing it occasionally states incorrect interest rates and invents '
        'promotional terms that do not exist, and it keeps no record of which documents it used for a '
        'given answer. It is live to the public and customers are not told they are talking to an AI.'
    ),
    'Hiring / resume screening': (
        'Our HR team uses a machine-learning model to screen and rank job applicants for shortlisting. '
        'It was trained on ten years of historical hiring decisions. We have not evaluated it for '
        'disparate impact across gender, age, or ethnicity, applicants are not informed that an '
        'automated system ranks them, and there is no human review before interview decisions. It is '
        'used for hiring in the EU.'
    ),
    'Credit underwriting model': (
        'We deployed a credit-underwriting model that approves or declines consumer loan applications. '
        'It went into production without documented validation of conceptual soundness, we did not test '
        'outcomes for fairness across protected classes, and there is no ongoing monitoring of '
        'performance or drift after deployment.'
    ),
}

scenario = SCENARIOS['GenAI banking assistant']
print(scenario)

## 5. Retrieve & generate a response

First the **applicable provisions** (retrieval + the graph): each with the risks it addresses and its cited cross-framework references. Then the **interpretation**: Regulus sends the LLM this *structured* context — provisions **plus their relationships** — and asks for a grounded, cited answer (assessment → applicable provisions & why → cross-framework view → next steps → sources).

> With no API key, `answer()` runs in **dry-run** mode and shows the exact structured context that would be sent — so the pipeline is inspectable offline. For real generation: `pip install openai` and set `REGULUS_RETRIEVER=embedding` (keys are reused from your GKN `.env`).

In [ ]:
reg.lookup(scenario, top_k=6)

In [ ]:
_ = reg.answer(scenario)

## 6. Visualize the regulatory neighborhood

The local map of the issue: its direct-hit provisions (bold-outlined squares), the cited crosswalk provisions they link to (squares, colored by framework), and the risk categories they address (grey circles).

In [ ]:
_ = reg.visualize(scenario, top_k=3)   # needs matplotlib

## What's next

- **Sharper retrieval** — set `RETRIEVER = 'embedding'` above for embedding-quality context.
- **Evidence paths** — return the full trail (issue → provision → crosswalk → provision) via GKN's path explainer.
- **Measure the claim** — benchmark the graph-grounded answer against a flat-RAG baseline.
- **Interface** — a "submit an issue" web UI.

**Extend Regulus:** add crosswalk rows or frameworks under `data/`, or swap the seed crosswalks for authoritative mappings.